In [42]:
# evaluate_yolov11_final.py
# Full evaluation script that mirrors your working inference pipeline (letterbox, dtype handling, scaling).
# Saves COCO-format predictions and runs COCOeval.
#
# Usage: place in repo root and run: python evaluate_yolov11_final.py

import os
import cv2
import json
import csv
import torch
import gc
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from time import perf_counter

In [43]:
# ---- CONFIG ----
WEIGHT = "internal_assets/weights/best.pt"

IMG_DIR = "COCO/images/val"                       # folder with images (must match COCO file_name)
ANN_FILE = "COCO/annotations/instances_val.json"


IMG_SIZE = (640, 640)   # (H, W) as used by your model/training
CONF_THR = 0.55
IOU_THR = 0.45
SAVE_JSON = "yolov11_predictions.json"
VISUALIZE = False        # True -> save a few debug images (vis_<imgid>.jpg)
# ----------------

# COCO 0..79 -> 1..90 mapping (use only if annotations require it)
COCO91CLASS = {
    0:1,1:2,2:3,3:4,4:5,5:6,6:7,7:8,8:9,9:10,
    10:11,11:13,12:14,13:15,14:16,15:17,16:18,17:19,18:20,
    19:21,20:22,21:23,22:24,23:25,24:27,25:28,26:31,27:32,
    28:33,29:34,30:35,31:36,32:37,33:38,34:39,35:40,36:41,
    37:42,38:43,39:44,40:46,41:47,42:48,43:49,44:50,45:51,
    46:52,47:53,48:54,49:55,50:56,51:57,52:58,53:59,54:60,
    55:61,56:62,57:63,58:64,59:65,60:67,61:70,62:72,63:73,
    64:74,65:75,66:76,67:77,68:78,69:79,70:80,71:81,72:82,
    73:84,74:85,75:86,76:87,77:88,78:89,79:90
}

In [44]:
# ---- helpers (reuse exact letterbox from your working inference) ----
def letterbox(img, new_shape=(640, 640), color=(114, 114, 114)):
    """Resize and pad image (returns padded_img, gain, (pad_left, pad_top))."""
    h0, w0 = img.shape[:2]
    new_h, new_w = new_shape
    r = min(new_h / h0, new_w / w0)
    new_unpad_w = int(round(w0 * r))
    new_unpad_h = int(round(h0 * r))
    img_resized = cv2.resize(img, (new_unpad_w, new_unpad_h), interpolation=cv2.INTER_LINEAR)
    dw = new_w - new_unpad_w
    dh = new_h - new_unpad_h
    top = int(round(dh / 2 - 0.1))
    bottom = int(round(dh / 2 + 0.1))
    left = int(round(dw / 2 - 0.1))
    right = int(round(dw / 2 + 0.1))
    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right,
                                    cv2.BORDER_CONSTANT, value=color)
    return img_padded, r, (left, top)

def scale_coords_from_padded(dets, gain, pad, orig_w, orig_h):
    """
    dets: tensor Nx6 (x1,y1,x2,y2,conf,cls) in padded image coordinates
    gain: scaling gain used for resize
    pad: (pad_left, pad_top) values returned by letterbox
    """
    # operate inplace on a clone outside if needed
    dets = dets.clone()
    pad_left, pad_top = pad
    # remove padding
    dets[:, [0, 2]] -= pad_left
    dets[:, [1, 3]] -= pad_top
    # divide by gain (undo scaling)
    dets[:, :4] /= gain
    # clamp
    dets[:, [0, 2]] = dets[:, [0, 2]].clamp(0, orig_w)
    dets[:, [1, 3]] = dets[:, [1, 3]].clamp(0, orig_h)
    return dets

# ---- import repo NMS (assumes utils/util.py in repo) ----
try:
    from utils.util import non_max_suppression  # expects model-native output shape [B, C, N]
except Exception as e:
    raise RuntimeError("Failed to import repo non_max_suppression from src.utils.util: " + str(e))

OUT_DIR = "evaluation_visualize/base_result"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- main evaluation ----
def main():
    # --- robust model load / move / dtype handling ---
    ckpt = torch.load(WEIGHT, map_location="cpu", weights_only=False)
    if 'model' not in ckpt:
        raise RuntimeError("Checkpoint does not contain 'model' key.")
    model = ckpt['model']

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Selected device:", device)

    # 1) move model to device first
    model = model.to(device)

    # 2) choose dtype based on device (only use fp16 on CUDA)
    USE_FP16 = True  # set to False if you want float32 even on GPU
    if device.type == "cuda" and USE_FP16:
        model = model.half()
    else:
        model = model.float()

    # 3) move any plain-Tensor attributes (not registered buffers) to device/dtype
    #    This handles cases where model authors stored tensors as attributes instead of register_buffer.
    target_dtype = torch.float16 if (device.type == "cuda" and USE_FP16) else torch.float32
    for module in model.modules():
        for name, val in list(vars(module).items()):
            if isinstance(val, torch.Tensor):
                # Only convert if needed
                if val.device != device or val.dtype != target_dtype:
                    new_val = val.to(device).to(target_dtype)
                    setattr(module, name, new_val)

    # 4) final eval and a short sanity print
    model.eval()

    # Sanity check: print a compact device/dtype summary
    from collections import defaultdict
    dev_counts = defaultdict(int)
    for p in model.parameters():
        dev_counts[(str(p.device), str(p.dtype))] += p.numel()
    print("Param device/dtype breakdown AFTER move:")
    for k, v in dev_counts.items():
        print(" ", k, v)

    buf_counts = defaultdict(int)
    for n, b in model.named_buffers():
        buf_counts[(str(b.device), str(b.dtype))] += b.numel()
    print("Buffer device/dtype breakdown AFTER move:")
    for k, v in buf_counts.items():
        print(" ", k, v)


    cocoGt = COCO(ANN_FILE)
    img_ids = cocoGt.getImgIds()

    results = []
    processed = 0


    # timing collection
    per_image_timings = []  # list of dicts
    eval_start_ts = perf_counter()

    for img_id in tqdm(img_ids, desc="Evaluating", unit="img"):
        info = cocoGt.loadImgs(img_id)[0]
        file_name = info['file_name']
        img_path = os.path.join(IMG_DIR, file_name)
        if not os.path.exists(img_path):
            # skip if image not available locally
            continue

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        orig_h, orig_w = img_bgr.shape[:2]

        # preprocessing (exact same as your working inference)
        img_pad, gain, (pad_w, pad_h) = letterbox(img_bgr, new_shape=IMG_SIZE)
        img_rgb = cv2.cvtColor(img_pad, cv2.COLOR_BGR2RGB)
        img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        img_tensor = img_tensor.to(device)
        if next(model.parameters()).dtype == torch.half:
            img_tensor = img_tensor.half()

        # --- forward timing ---
        # warmup sync is not done per image here; measured per-image
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_forward0 = perf_counter()
        with torch.no_grad():
            out = model(img_tensor)
            if isinstance(out, (list, tuple)):
                out = out[0]
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_forward1 = perf_counter()
        forward_ms = (t_forward1 - t_forward0) * 1000.0

        # out expected shape: [B, C, N]
        if not torch.is_tensor(out):
            # record a timing entry with zeros and skip
            per_image_timings.append({
                "image_id": img_id,
                "forward_ms": forward_ms,
                "nms_ms": 0.0,
                "scale_coords_ms": 0.0,
                "total_ms": forward_ms,
                "pre_nms_count": 0,
                "post_nms_count": 0,
            })
            continue

        # Ensure class logits are probabilities: if raw logits exceed 1 apply sigmoid to class slice
        try:
            C = out.shape[1]
            nc = C - 4
            cls_slice = out[:, 4:4+nc]
            if float(cls_slice.max()) > 1.0:
                out[:, 4:4+nc] = cls_slice.sigmoid()
        except Exception:
            nc = max(0, out.shape[1] - 4)  # fallback

        # Count pre-NMS candidates (heuristic): number of anchors/positions whose max class score > CONF_THR
        # We permute to [B, N, C] to operate per candidate
        try:
            out_perm = out.permute(0, 2, 1)  # [B, N, C]
            if out_perm.shape[-1] >= 5 and nc > 0:
                # class probs slice:
                class_slice = out_perm[:, :, 4:4+nc]  # [B, N, nc]
                # per-candidate best class score
                best_scores, _ = class_slice.max(dim=2)  # [B, N]
                pre_nms_count = int((best_scores > CONF_THR).sum().item())
            else:
                # fallback: if format unknown, use simple threshold on 5th channel
                scores_chan = out_perm[:, :, 4] if out_perm.shape[-1] > 4 else out_perm[:, :, -1]
                pre_nms_count = int((scores_chan > CONF_THR).sum().item())
        except Exception:
            pre_nms_count = -1  # unknown

        # Run repo NMS (works with output as-is) and time it
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_nms0 = perf_counter()
        dets = non_max_suppression(out, confidence_threshold=CONF_THR, iou_threshold=IOU_THR)[0]
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_nms1 = perf_counter()
        nms_ms = (t_nms1 - t_nms0) * 1000.0

        post_nms_count = 0
        if dets is None or len(dets) == 0:
            post_nms_count = 0
            # still record timing and continue (no detections)
            per_image_timings.append({
                "image_id": img_id,
                "forward_ms": forward_ms,
                "nms_ms": nms_ms,
                "scale_coords_ms": 0.0,
                "total_ms": forward_ms + nms_ms,
                "pre_nms_count": pre_nms_count,
                "post_nms_count": post_nms_count,
            })
            processed += 1
            continue

        # Map boxes from padded input -> original image exactly like inference
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_scale0 = perf_counter()
        dets = dets.detach().cpu()
        dets = scale_coords_from_padded(dets, gain, (pad_w, pad_h), orig_w, orig_h)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t_scale1 = perf_counter()
        scale_ms = (t_scale1 - t_scale0) * 1000.0

        post_nms_count = int(len(dets))

        # collect results (COCO expects [x,y,w,h] and category IDs must match annotation file)
        gt_cat_ids = set([c['id'] for c in cocoGt.loadCats(cocoGt.getCatIds())])
        mapping_needed = (0 not in gt_cat_ids)  # if GT IDs are 1..90, map from 0..79 to 1..90

        for *xyxy, conf, cls in dets:
            x1, y1, x2, y2 = [float(v) for v in xyxy]
            w = x2 - x1
            h = y2 - y1
            if w <= 0 or h <= 0:
                continue

            cls_idx = int(cls.item())  # YOLO class index (0..79)
            if mapping_needed:
                coco_cat_id = COCO91CLASS.get(cls_idx)
                if coco_cat_id is None:
                    continue
            else:
                coco_cat_id = cls_idx

            score = float(conf.item())
            score = max(0.0, min(1.0, score))

            results.append({
                "image_id": img_id,
                "category_id": int(coco_cat_id),
                "bbox": [max(0.0, x1), max(0.0, y1), float(w), float(h)],
                "score": score
            })

        # optional visualization of one image (quick sanity)
        if VISUALIZE and processed % 10 == 0:
            vis = img_bgr.copy()
            for *xyxy, conf, cls in dets:
                xa, ya, xb, yb = map(int, xyxy)
                cv2.rectangle(vis, (xa, ya), (xb, yb), (0,255,0), 2)
                cv2.putText(vis, f"{int(cls.item())}:{conf.item():.2f}", (xa, max(0, ya-6)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            cv2.imwrite(os.path.join(OUT_DIR, f"vis_{img_id}.jpg"), vis)

        processed += 1

        total_ms = forward_ms + nms_ms + scale_ms
        per_image_timings.append({
            "image_id": img_id,
            "forward_ms": forward_ms,
            "nms_ms": nms_ms,
            "scale_coords_ms": scale_ms,
            "total_ms": total_ms,
            "pre_nms_count": pre_nms_count,
            "post_nms_count": post_nms_count,
        })

    eval_end_ts = perf_counter()
    eval_elapsed_s = eval_end_ts - eval_start_ts

    # save predictions
    with open(SAVE_JSON, "w") as f:
        json.dump(results, f)
    print(f"\nSaved {len(results)} predictions to {SAVE_JSON} (processed {processed} images)")

    # run COCOeval
    if len(results) == 0:
        print("No results to evaluate.")
        return

    cocoDt = cocoGt.loadRes(SAVE_JSON)
    cocoEval = COCOeval(cocoGt, cocoDt, "bbox")
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()

    # save coco summary + metadata (unchanged)
    stat_names = [
        "AP_IoU=0.50:0.95_area=all_maxDets=100",
        "AP_IoU=0.50_area=all_maxDets=100",
        "AP_IoU=0.75_area=all_maxDets=100",
        "AP_IoU=0.50:0.95_area=small_maxDets=100",
        "AP_IoU=0.50:0.95_area=medium_maxDets=100",
        "AP_IoU=0.50:0.95_area=large_maxDets=100",
        "AR_IoU=0.50:0.95_area=all_maxDets=1",
        "AR_IoU=0.50:0.95_area=all_maxDets=10",
        "AR_IoU=0.50:0.95_area=all_maxDets=100",
        "AR_IoU=0.50:0.95_area=small_maxDets=100",
        "AR_IoU=0.50:0.95_area=medium_maxDets=100",
        "AR_IoU=0.50:0.95_area=large_maxDets=100",
    ]

    stats = cocoEval.stats if hasattr(cocoEval, "stats") else None
    summary_vals = {name: (None if stats is None or np.isnan(stats[i]) else float(stats[i]))
                    for i, name in enumerate(stat_names)}

    metadata = {
        "conf_threshold": float(CONF_THR),
        "iou_thrs": cocoEval.params.iouThrs.tolist(),
        "area_ranges": getattr(cocoEval.params, "areaRng", None),
        "num_images_processed": len(cocoEval.params.imgIds) if hasattr(cocoEval.params, "imgIds") else None,
        "number_of_predictions": len(results),
    }

    summary_with_meta = {
        "summary": summary_vals,
        "metadata": metadata,
        "evaluation_times": float(eval_elapsed_s)
    }

    out_summary_path = os.path.join(OUT_DIR, f"conf-{CONF_THR}/coco_summary_with_meta.json")
    with open(out_summary_path, "w") as f:
        json.dump(summary_with_meta, f, indent=2)
    print(f"Saved summary + metadata to {out_summary_path}")

    # Save per-image timings to CSV and JSON and NPZ
    csv_path = os.path.join(OUT_DIR, f"conf-{CONF_THR}/per_image_timings.csv")
    json_path = os.path.join(OUT_DIR, f"conf-{CONF_THR}/per_image_timings.json")
    npz_path = os.path.join(OUT_DIR, f"conf-{CONF_THR}/per_image_timings.npz")

    # CSV
    keys = ["image_id", "forward_ms", "nms_ms", "scale_coords_ms", "total_ms", "pre_nms_count", "post_nms_count"]
    with open(csv_path, "w", newline="") as cf:
        writer = csv.DictWriter(cf, fieldnames=keys)
        writer.writeheader()
        for row in per_image_timings:
            writer.writerow({k: row.get(k, "") for k in keys})

    # JSON
    with open(json_path, "w") as jf:
        json.dump({
            "per_image_timings": per_image_timings,
            "evaluation_times": summary_with_meta["evaluation_times"]
        }, jf, indent=2)

    # NPZ (arrays)
    # convert lists to arrays (fill missing)
    forward_arr = np.array([row.get("forward_ms", np.nan) for row in per_image_timings], dtype=float)
    nms_arr = np.array([row.get("nms_ms", np.nan) for row in per_image_timings], dtype=float)
    scale_arr = np.array([row.get("scale_coords_ms", np.nan) for row in per_image_timings], dtype=float)
    total_arr = np.array([row.get("total_ms", np.nan) for row in per_image_timings], dtype=float)
    pre_arr = np.array([row.get("pre_nms_count", -1) for row in per_image_timings], dtype=int)
    post_arr = np.array([row.get("post_nms_count", -1) for row in per_image_timings], dtype=int)

    np.savez_compressed(npz_path,
                        forward_ms=forward_arr,
                        nms_ms=nms_arr,
                        scale_ms=scale_arr,
                        total_ms=total_arr,
                        pre_nms_count=pre_arr,
                        post_nms_count=post_arr,
                        eval_elapsed_s=eval_elapsed_s)

    print(f"Saved timing CSV to {csv_path}")
    print(f"Saved timing JSON to {json_path}")
    print(f"Saved timing NPZ to {npz_path}")

    # Save detailed eval arrays (.npz) as you already do
    eval_dict = getattr(cocoEval, "eval", None)
    if eval_dict is not None:
        precision = eval_dict.get("precision")  # shape [T, R, K, A, M]
        recall = eval_dict.get("recall")        # shape [T, K, A, M] or similar
        scores = eval_dict.get("scores")
        np.savez_compressed(
            os.path.join(OUT_DIR, f"conf-{CONF_THR}/coco_eval_details.npz"),
            precision=precision if precision is not None else np.array([]),
            recall=recall if recall is not None else np.array([]),
            scores=scores if scores is not None else np.array([]),
            metadata_json=json.dumps(metadata)  # store metadata string for convenience
        )
        print(f"Saved detailed eval arrays to conf-{CONF_THR}/coco_eval_details.npz")
    else:
        print("No cocoEval.eval data to save.")

In [45]:
if __name__ == "__main__":
    main()


Selected device: cpu
Param device/dtype breakdown AFTER move:
  ('cpu', 'torch.float32') 2624080
Buffer device/dtype breakdown AFTER move:
  ('cpu', 'torch.float32') 15664
  ('cpu', 'torch.int64') 81
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


Evaluating: 100%|██████████| 20/20 [00:02<00:00,  7.59img/s]



Saved 19 predictions to yolov11_predictions.json (processed 20 images)
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.043
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.046
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.046
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.027
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.080
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.036
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.061
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.